In [1]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

from ksw import Cosmology, utils
from ksw import estimator
import camb

from ksw import Afunctionals

In [2]:
# Setup CAMB parameters
pars = camb.CAMBparams()
pars.WantTensors=True
pars.set_cosmology(H0=67.66, ombh2=0.02242, omch2=0.11933)
pars.InitPower.set_params(As=2.1056e-9, ns=0.9665, r=0.001)
cosmo = Cosmology(pars, verbose=False)

# Compute transfer functions
print("Computing transfer functions...")
cosmo.compute_transfer(lmax=300)

# Compute angular power spectra
print("Computing angular power spectra...")
#cosmo.compute_c_ell()

Computing transfer functions...
Computing angular power spectra...


In [3]:
from ksw.shape import Shape
radii = np.logspace(0, 3, 10)  # Small number of radii for quick testing

prim_shape = Shape.prim_local(ns=0.9665)
radii = np.logspace(0, 3, 10)

cosmo.compute_transfer(lmax=300)
cosmo.add_prim_reduced_bispectrum_scalar_dL(prim_shape, radii)

cosmo.compute_transfer_tensor(lmax=300)
cosmo.add_prim_reduced_bispectrum_tensor_dL(prim_shape, radii)

Updated CAMB param: WantTensors from False to True.
Updated CAMB param: WantTensors from True to False.


In [4]:
# Create a estimator instance
red_bispectra = cosmo.red_bispectra
icov = lambda alm: alm
pol = ('T', 'E', 'B')
estimator_con = estimator.KSW(red_bispectra, icov, lmax=100, pol=pol)

In [5]:
import healpy as hp
lmax = estimator_con.lmax
nelem = hp.Alm.getsize(lmax+1)
print(nelem)

5253


In [6]:
# Generate sample alm data for testing
lmax = estimator_con.lmax
npol = 3
#nelem = (lmax + 1) * (lmax + 2) // 2   # HEALPix alm element count
nelem = hp.Alm.getsize(lmax)

# Create random alm with correct shape
#np.random.seed(42)
alm_test = np.random.randn(npol, nelem) + 1j * np.random.randn(npol, nelem)

# Try to call compute_estimate_sst
L_list = np.linspace(0, 100, 101) 
Lmax = 100 

estimate, cubic, lin_term, fisher = estimator_con.compute_estimate_sst(
    alm_test,
    L_list,
    Lmax,
    theta_batch=25
)

(20, 3, 2, 101)
0.0
fnl=np.float64(nan), t_cubic=np.float64(nan), lin_term=0, fisher=1
